# Day 35 - LIS and LCS: two longest-subsequence problems that only look alike

Day 33 built dynamic programming out of one-dimensional tables and day 34 made
the table two-dimensional.  Today both shapes show up again on two problems that
are constantly mentioned in the same breath - longest increasing subsequence and
longest common subsequence - and then diverge.

LIS has a quadratic DP that can be deleted outright: patience sorting plus the
`lower_bound` from day 23 answers it in `O(n log n)`.  LCS has no such shortcut
in general.  But when the two inputs are permutations of one another, LCS *is*
LIS, and inherits the speedup - which is where a `git diff` and the ROUGE-L
metric come from.

In [1]:
from bisect import bisect_left, bisect_right
import random
import time

## 1. LIS the honest way: `O(n^2)`

The state has to be "the longest increasing subsequence **ending at** i".  The
more natural phrasing - "the LIS of the first i elements" - is not enough: to
extend a subsequence you must know what its last element was, and that phrasing
has already thrown it away.

In [2]:
def lis_dp(a):
    """dp[i] = length of the longest increasing subsequence *ending at* i.

    "Ending at i" is the whole design decision.  The natural phrasing - "the
    LIS of the first i elements" - is not enough state: to extend a
    subsequence you have to know what its last element was, and that phrasing
    has thrown it away.
    """
    n = len(a)
    if n == 0:
        return 0, []
    dp = [1] * n
    parent = [-1] * n
    for i in range(n):
        for j in range(i):
            if a[j] < a[i] and dp[j] + 1 > dp[i]:
                dp[i] = dp[j] + 1
                parent[i] = j
    best = max(range(n), key=lambda i: dp[i])
    out = []
    while best != -1:
        out.append(a[best])
        best = parent[best]
    out.reverse()
    return max(dp), out

A = [10, 9, 2, 5, 3, 7, 101, 18]
n, seq = lis_dp(A)
print('a       =', A)
print('length  =', n)
print('one LIS =', seq)

a       = [10, 9, 2, 5, 3, 7, 101, 18]
length  = 4
one LIS = [2, 5, 7, 101]


## 2. Patience sorting: deal the cards, count the piles

Keep a single array `tails`, where `tails[k]` is the smallest value that can end
an increasing subsequence of length `k+1` seen so far.  It is sorted by
construction, so the slot for each incoming value is a binary search - the
`lower_bound` of day 23.  Each element is placed once, so the whole run costs
`O(n log n)`.

The card game the algorithm is named after is the same thing: each pile's top
card is a `tails` entry, and the number of piles is the answer.

In [3]:
def lis_patience(a, strict=True):
    """tails[k] = the smallest value that can end an increasing subsequence
    of length k + 1 seen so far.

    tails is sorted by construction, so the position of the incoming value is
    a `lower_bound` (day 23).  `bisect_left` refuses to sit after an equal
    value and therefore builds a *strictly* increasing subsequence;
    `bisect_right` allows it and builds a non-decreasing one.  One function
    name is the entire difference between the two problems.
    """
    find = bisect_left if strict else bisect_right
    tails = []
    for x in a:
        k = find(tails, x)
        if k == len(tails):
            tails.append(x)          # x extends the longest run so far
        else:
            tails[k] = x             # x is a cheaper ending for length k+1
    return len(tails), tails

def patience_piles(a):
    """The card game the algorithm is named after: each pile's top card is a
    tails entry, and the number of piles is the answer."""
    piles = []
    for x in a:
        k = bisect_left([p[-1] for p in piles], x)
        if k == len(piles):
            piles.append([x])
        else:
            piles[k].append(x)
    return piles

ln, tails = lis_patience(A)
print('tails  =', tails, ' length', ln)
for k, pile in enumerate(patience_piles(A)):
    print(f'pile {k}: {pile}')

tails  = [2, 3, 7, 18]  length 4
pile 0: [10, 9, 2]
pile 1: [5, 3]
pile 2: [7]
pile 3: [101, 18]


## 3. `tails` is a scoreboard, not an answer

This is the trap.  `tails` always has the right *length*, and usually holds
values that never formed a subsequence in that order - the last entry can even
be an element that arrived last in the input.  The real subsequence has to be
recovered from parent pointers recorded at insert time.

In [4]:
def lis_patience_reconstruct(a):
    """The same walk, but recording who came before whom.

    This is the trap the tails array sets: `tails` has the right *length* and
    is almost never a subsequence of the input.  The actual answer has to be
    recovered from parent pointers taken at insert time.
    """
    tails, idx, parent = [], [], [-1] * len(a)
    for i, x in enumerate(a):
        k = bisect_left(tails, x)
        if k == len(tails):
            tails.append(x)
            idx.append(i)
        else:
            tails[k] = x
            idx[k] = i
        parent[i] = idx[k - 1] if k else -1
    out, i = [], (idx[-1] if idx else -1)
    while i != -1:
        out.append(a[i])
        i = parent[i]
    out.reverse()
    return len(tails), out, tails

trap = [2, 6, 8, 3, 4, 5, 1]
print('a        =', trap)
print('tails    =', lis_patience(trap)[1], ' <- 1 is the last element of a')
print('real LIS =', lis_patience_reconstruct(trap)[1])

a        = [2, 6, 8, 3, 4, 5, 1]
tails    = [1, 3, 4, 5]  <- 1 is the last element of a
real LIS = [2, 3, 4, 5]


## 4. Strictly increasing vs non-decreasing is one function name

`bisect_left` refuses to sit after an equal value, so an equal value overwrites
the same slot and the run stays *strictly* increasing.  `bisect_right` lets it
move one slot further, which allows repeats.  Nothing else in the algorithm
changes.

In [5]:
dup = [1, 3, 3, 3, 5]
print('a                       =', dup)
print('bisect_left  (strict)   =', lis_patience(dup, strict=True)[0])
print('bisect_right (allows =) =', lis_patience(dup, strict=False)[0])

a                       = [1, 3, 3, 3, 5]
bisect_left  (strict)   = 3
bisect_right (allows =) = 5


## 5. LeetCode 354 - Russian Doll Envelopes

**The task.** You are given envelopes, each with a width and a height.  One
envelope fits inside another only if *both* its width and its height are
strictly smaller.  Return the largest number of envelopes you can nest.

**Input / output.** `envelopes[i] = [w, h]`; return an integer.

**Example.** `[[5,4],[6,4],[6,7],[2,3]]` -> `3`, nesting `[2,3]` in `[5,4]` in
`[6,7]`.

**Constraints.** Up to `10^5` envelopes, so an `O(n^2)` pass is too slow - the
intended solution is the `O(n log n)` LIS.

[leetcode.com/problems/russian-doll-envelopes](https://leetcode.com/problems/russian-doll-envelopes/)

Sort by width ascending and, inside equal widths, by height **descending**; then
take the LIS of the heights.  The descending tie-break is the whole problem:
equal widths cannot nest, and a descending height order makes it impossible for
two envelopes of the same width to both appear in an increasing run.

In [6]:
def max_envelopes(envelopes):
    """Sort width ascending, height *descending* inside equal widths, then
    take the LIS of the heights.

    The descending tie-break is the whole problem.  Equal widths cannot nest,
    and a descending height order guarantees that no two envelopes of the same
    width can both appear in an increasing run of heights.  Sort both keys
    ascending and [(1,1),(1,2)] silently becomes a valid pair of dolls.
    """
    envelopes.sort(key=lambda e: (e[0], -e[1]))
    return lis_patience([h for _, h in envelopes])[0]

env = [[5, 4], [6, 4], [6, 7], [2, 3]]
print('sorted :', sorted(env, key=lambda e: (e[0], -e[1])))
print('answer :', max_envelopes(env))
print('all width 1:', max_envelopes([[1, 1], [1, 2], [1, 3]]), '(cannot nest)')

sorted : [[2, 3], [5, 4], [6, 7], [6, 4]]
answer : 3
all width 1: 1 (cannot nest)


## 6. LeetCode 673 - Number of Longest Increasing Subsequence

**The task.** Given an integer array, return how many *different* increasing
subsequences have the maximum length.

**Input / output.** An array `nums`; return an integer count.

**Example.** `[1,3,5,4,7]` -> `2`: the LIS length is 4, achieved by `1,3,5,7`
and `1,3,4,7`.

**Constraints.** Up to 2000 elements, so the quadratic DP is fine; the answer
fits in a 32-bit integer.

[leetcode.com/problems/number-of-longest-increasing-subsequence](https://leetcode.com/problems/number-of-longest-increasing-subsequence/)

Carry a second table alongside `dp`.  When a strictly longer subsequence is
found the count is **replaced**; when an equally long one is found the counts are
**added**.  Mixing those two up is the classic failure here.

In [7]:
def count_lis(a):
    """Carry a second table: cnt[i] = how many LIS of length dp[i] end at i.

    When a longer subsequence is found the count is *replaced*; when an
    equally long one is found the counts are *added*.  Mixing those two up is
    the classic off-by-a-lot in this problem.
    """
    n = len(a)
    if n == 0:
        return 0
    dp = [1] * n
    cnt = [1] * n
    for i in range(n):
        for j in range(i):
            if a[j] < a[i]:
                if dp[j] + 1 > dp[i]:
                    dp[i] = dp[j] + 1
                    cnt[i] = cnt[j]
                elif dp[j] + 1 == dp[i]:
                    cnt[i] += cnt[j]
    longest = max(dp)
    return sum(c for l, c in zip(dp, cnt) if l == longest)

print(count_lis([1, 3, 5, 4, 7]), 'for [1,3,5,4,7]')
print(count_lis([2, 2, 2, 2, 2]), 'for [2,2,2,2,2]')

2 for [1,3,5,4,7]
5 for [2,2,2,2,2]


## 7. LCS - the two-dimensional table

`dp[i][j]` is the LCS length of the first `i` characters of `x` and the first
`j` of `y`.  Two cases only.  If the last characters match they can always be
used together, so the answer is `1 +` the diagonal.  If they do not match, one of
the two is useless and we do not know which, so take the better of dropping each.

In [8]:
def lcs_table(x, y):
    """dp[i][j] = LCS length of x[:i] and y[:j].

    Two cases only.  If the last characters match, they *must* be usable
    together - no optimal solution is hurt by taking them - so the answer is
    1 + the diagonal.  If they do not match, one of the two last characters is
    useless, and we do not know which, so we take the better of dropping each.
    """
    n, m = len(x), len(y)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if x[i - 1] == y[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp

def lcs_backtrack(x, y, dp=None):
    """Walk the finished table backwards from the bottom-right corner."""
    dp = dp or lcs_table(x, y)
    i, j, out = len(x), len(y), []
    while i and j:
        if x[i - 1] == y[j - 1]:
            out.append(x[i - 1])
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    out.reverse()
    return ''.join(out) if isinstance(x, str) else out

x, y = 'AGGTAB', 'GXTXAYB'
dp = lcs_table(x, y)
print('    ' + ' '.join(f'{c:>3s}' for c in '-' + y))
for i, row in enumerate(dp):
    print(f'{"-" if i == 0 else x[i-1]}   ' + ' '.join(f'{v:3d}' for v in row))
print('\nlength =', dp[-1][-1])
print('LCS    =', lcs_backtrack(x, y, dp))

      -   G   X   T   X   A   Y   B
-     0   0   0   0   0   0   0   0
A     0   0   0   0   0   1   1   1
G     0   1   1   1   1   1   1   1
G     0   1   1   1   1   1   1   1
T     0   1   1   2   2   2   2   2
A     0   1   1   2   2   3   3   3
B     0   1   1   2   2   3   3   4

length = 4
LCS    = GTAB


## 8. Rolling rows: the number survives, the path does not

Each cell reads only the row above and the cell to its left, so two rows are
enough - `O(min(n, m))` space.  Backtracking needs the whole table, so this
version can only ever answer "how long", exactly the trade the 1-D knapsack made
on day 34.

In [9]:
def lcs_rolling(x, y):
    """Two rows instead of n+1 - O(min(n, m)) space.

    The length survives; the path does not.  Backtracking needs the whole
    table, so the rolling-row version can only ever answer "how long".  Same
    trade as the 1-D knapsack on day 34.
    """
    if len(y) > len(x):
        x, y = y, x
    prev = [0] * (len(y) + 1)
    for i in range(1, len(x) + 1):
        cur = [0] * (len(y) + 1)
        for j in range(1, len(y) + 1):
            cur[j] = prev[j - 1] + 1 if x[i - 1] == y[j - 1] else max(prev[j], cur[j - 1])
        prev = cur
    return prev[-1]

print('rolling:', lcs_rolling(x, y), '(same number, no path)')

rolling: 4 (same number, no path)


## 9. LCS of two permutations *is* LIS

If every element is distinct and both sequences hold the same set, relabel the
second one by "where does this value sit in the first".  A common subsequence is
then exactly an increasing run of those labels - so the answer is an LIS, and the
cost drops from `O(n*m)` to `O(n log n)`.  This is the small case of the
Hunt-Szymanski algorithm, which does the same with match lists when values repeat.

In [10]:
def lcs_of_permutations(p, q):
    """When every element is distinct and both lists hold the same set,
    relabel q by "where does this value sit in p".

    A common subsequence of p and q is then exactly an increasing run of those
    labels, so the answer is an LIS - O(n log n) instead of O(n*m).  This is
    the small case of the Hunt-Szymanski algorithm, which does the same trick
    with match lists when values repeat.
    """
    pos = {v: i for i, v in enumerate(p)}
    mapped = [pos[v] for v in q if v in pos]
    length, seq, _ = lis_patience_reconstruct(mapped)
    return length, [p[i] for i in seq]

p = [1, 2, 3, 4, 5, 6]
q = [2, 4, 1, 5, 6, 3]
print('index in p :', [p.index(v) for v in q])
print('LCS via LIS:', lcs_of_permutations(p, q))

random.seed(35)
sz = 1200
pp = list(range(sz)); random.shuffle(pp)
qq = list(range(sz)); random.shuffle(qq)
t0 = time.perf_counter(); slow = lcs_rolling(pp, qq); t1 = time.perf_counter()
fast = lcs_of_permutations(pp, qq)[0];                 t2 = time.perf_counter()
print(f'\nn = {sz} random permutations')
print(f'  O(n*m) table   : {slow} in {t1-t0:.3f}s')
print(f'  O(n log n) LIS : {fast} in {t2-t1:.4f}s  ({(t1-t0)/(t2-t1):.0f}x faster)')

index in p : [1, 3, 0, 4, 5, 2]
LCS via LIS: (4, [2, 4, 5, 6])

n = 1200 random permutations
  O(n*m) table   : 67 in 0.725s
  O(n log n) LIS : 67 in 0.0014s  (502x faster)


## 10. LeetCode 1035 - Uncrossed Lines

**The task.** Two rows of integers are written down.  You may connect equal
numbers with straight lines, but no two lines may cross and each number is used
at most once.  Return the largest number of lines.

**Input / output.** Two arrays `nums1`, `nums2`; return an integer.

**Example.** `nums1 = [1,4,2]`, `nums2 = [1,2,4]` -> `2`: connect the two 1s and
then either the 4s or the 2s, but not both, because those two lines would cross.

**Constraints.** Both arrays are at most 500 long, so the `O(n*m)` table is
comfortable.

[leetcode.com/problems/uncrossed-lines](https://leetcode.com/problems/uncrossed-lines/)

Non-crossing means the chosen pairs keep their relative order in both rows - and
that is the definition of a common subsequence.  The picture is different; the
recurrence is character for character the LCS one.

In [11]:
def max_uncrossed_lines(a, b):
    """Non-crossing pairs of equal numbers = a common subsequence.  The
    picture is different; the recurrence is character for character the LCS
    one."""
    return lcs_rolling(a, b)

print(max_uncrossed_lines([1, 4, 2], [1, 2, 4]))

2


## 11. What LCS is actually for: diff and ROUGE-L

Everything outside the LCS of two files is either a deletion or an insertion -
that is what `git diff` means by "unchanged" (it uses Myers' algorithm, but the
same definition).  And ROUGE-L, the summarisation metric, is an F-measure over
the LCS length of the candidate and the reference.

In [12]:
def diff(old, new):
    """A minimal line diff: everything outside the LCS is a delete or an add.

    This is what `git diff` is doing underneath (with a better algorithm -
    Myers - but the same definition of "unchanged").
    """
    dp = lcs_table(old, new)
    i, j, out = len(old), len(new), []
    while i or j:
        if i and j and old[i - 1] == new[j - 1]:
            out.append(('  ', old[i - 1])); i -= 1; j -= 1
        elif j and (i == 0 or dp[i][j - 1] >= dp[i - 1][j]):
            out.append(('+ ', new[j - 1])); j -= 1
        else:
            out.append(('- ', old[i - 1])); i -= 1
    out.reverse()
    return out

def rouge_l(candidate, reference):
    """The summarisation metric: F-measure over the LCS length.

    Every time a model summary is scored with ROUGE-L, this table is what runs.
    """
    c, r = candidate.split(), reference.split()
    l = lcs_rolling(c, r)
    if not l:
        return 0.0
    prec, rec = l / len(c), l / len(r)
    return 2 * prec * rec / (prec + rec)

old = ['def f(x):', '    y = x + 1', '    return y']
new = ['def f(x):', '    # bump', '    y = x + 2', '    return y']
for tag, line in diff(old, new):
    print(tag + line)
print()
print('ROUGE-L =', round(rouge_l('the cat sat on the mat',
                                 'the cat was sitting on the mat'), 3))

  def f(x):
-     y = x + 1
+     # bump
+     y = x + 2
      return y

ROUGE-L = 0.769


## 12. LeetCode 300 - Longest Increasing Subsequence

**The task.** Given an integer array, return the length of its longest strictly
increasing subsequence.  A subsequence keeps the original order but may skip
elements.

**Input / output.** An array `nums`; return an integer.

**Example.** `[10,9,2,5,3,7,101,18]` -> `4`, for instance `2,3,7,101`.

**Constraints.** Up to 2500 elements and the follow-up explicitly asks for an
`O(n log n)` solution - which is the patience-sorting version above.

[leetcode.com/problems/longest-increasing-subsequence](https://leetcode.com/problems/longest-increasing-subsequence/)

In [13]:
def length_of_lis(nums):
    return lis_patience(nums)[0]

print(length_of_lis(A), length_of_lis([7, 7, 7, 7]))

4 1


## 13. Checks

In [14]:
assert lis_dp(A)[0] == 4 and lis_patience(A)[0] == 4
ln, real, _ = lis_patience_reconstruct(A)
assert len(real) == ln and all(real[i] < real[i+1] for i in range(ln - 1))
assert lis_patience([1,3,3,3,5], True)[0] == 3
assert lis_patience([1,3,3,3,5], False)[0] == 5
assert max_envelopes([[5,4],[6,4],[6,7],[2,3]]) == 3
assert max_envelopes([[1,1],[1,2],[1,3]]) == 1
assert count_lis([1,3,5,4,7]) == 2 and count_lis([2,2,2,2,2]) == 5
assert lcs_backtrack('AGGTAB', 'GXTXAYB') == 'GTAB'
assert lcs_rolling('AGGTAB', 'GXTXAYB') == 4
assert lcs_of_permutations(p, q)[0] == lcs_rolling(p, q)
assert max_uncrossed_lines([1,4,2], [1,2,4]) == 2
assert 0.7 < rouge_l('the cat sat on the mat', 'the cat was sitting on the mat') < 0.9
print('all assertions passed')

all assertions passed
